# Absolute fitness is not neutral under phenotypic uncertainty

Three reproduction-rate pairs whose *difference* is fixed at 0.003 while the
absolute level shifts. Classical population genetics at fixed population size
predicts identical dynamics in all three. That holds for a deterministic
genotype-phenotype map (DGP, left) but fails for a probabilistic one
(PrGP, right).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from propgen import load_summary
from propgen.plotting import set_paper_style

set_paper_style()
FIGDIR = REPO / "figures"
FIGDIR.mkdir(exist_ok=True)

dgp = load_summary(REPO / "results" / "absfit" / "summary_dgp.npz")
prgp = load_summary(REPO / "results" / "absfit" / "summary_prgp.npz")
dgp, prgp

In [ ]:
def plot_map(summary, pairs, title, ax):
    Ng, Np = summary.shape
    for r1 in summary.values("r1"):
        panel = summary.select(r1=r1)
        mean, sem, cycles = panel["mean"], panel["sem"], panel["cycles"]
        for g, p in pairs:
            line, = ax.plot(cycles, mean[g, p], lw=2, alpha=0.85,
                            label=f"$r_1$={r1}, $g$={g}, $p$={p}")
            ax.fill_between(cycles, mean[g, p] - sem[g, p], mean[g, p] + sem[g, p],
                            color=line.get_color(), alpha=0.2)
    ax.set_title(title)
    ax.set_xlabel("Dilution cycle")
    ax.legend(fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
plot_map(dgp, [(0, 0), (1, 1)], "Deterministic map (DGP)", axes[0])
plot_map(prgp, [(g, p) for g in range(2) for p in range(2)],
         "Probabilistic map (PrGP)", axes[1])
axes[0].set_ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGDIR / "absfit.pdf")
plt.show()

Quantitatively: the spread of final frequencies across the three absolute
fitness levels. Under a deterministic map it is essentially zero; under a
probabilistic map it is not.

In [ ]:
for name, summary in (("DGP ", dgp), ("PrGP", prgp)):
    finals = np.array([summary.select(r1=r1)["mean"][..., -50:].mean(axis=-1).ravel()
                       for r1 in summary.values("r1")])
    print(f"{name}: max spread across absolute fitness levels = {np.ptp(finals, axis=0).max():.4f}")